# 6CS012 – Artificial Intelligence and Machine Learning
## Worksheet 5: Implementation of Convolutional Neural Network using Keras

**Prepared By:** Siman Giri *(Module Leader – 6CS012)*  
**Date:** March 23, 2025

---

## 1. Instructions

This exercise sheet will help you understand and implement Convolutional Neural Networks from scratch using Keras.

- This worksheet must be completed **individually**.
- All solutions must be written in **Jupyter Notebook** (Preferred).

### Learning Objectives

By the end of this tutorial, learners should be able to effectively train, evaluate, save, and reuse CNN models, providing a solid foundation for real-world machine learning projects.

- **Understand Model Compilation and Training**
- **Evaluate and Test Model Performance**
- **Make Predictions and Interpret Results**

---

## 2. Building an End-to-End Image Classifier with CNNs

A Convolutional Neural Network (CNN) has two important sections:

1. **Convolutional Layer** – Extracts spatial features from images using learnable filters.
2. **Fully Connected Layer** – Interprets extracted features and outputs class predictions.

---

## 3. Understanding the Convolutional Layer

A **convolutional layer** is a core building block of CNNs. It applies convolution operations to extract spatial features from input images or tensors. In Keras, convolutional layers are implemented using the `Conv2D` class.

### 3.1 Conv2D Class in Keras

Each neuron in a `Conv2D` layer applies a set of filters to local patches of the input data, capturing spatial hierarchies of features.

**Syntax:**
```python
from tensorflow.keras.layers import Conv2D

layer = Conv2D(
    filters,
    kernel_size,
    strides=(1, 1),
    padding="valid",
    activation=None,
    use_bias=True,
    kernel_initializer="glorot_uniform"
)
```

### Main Arguments

| Argument | Description |
|---|---|
| `filters` | Number of filters (feature detectors) in the layer. |
| `kernel_size` | Size of the convolutional kernel (e.g., `(3,3)` or `(5,5)`). |
| `strides` | Step size for traversing input data. Default: `(1,1)`. |
| `padding` | `"valid"` (no padding) or `"same"` (zero-padding). |
| `activation` | Activation function applied to the output. Default: `None`. |
| `use_bias` | Whether to include a bias term. Default: `True`. |
| `kernel_initializer` | Method to initialize filter weights. Default: `"glorot_uniform"`. |
| `bias_initializer` | Method to initialize the bias. Default: `"zeros"`. |
| `kernel_regularizer` | Regularization applied to filter weights (e.g., L1, L2). |
| `bias_regularizer` | Regularization applied to the bias. |

### 3.2 How Does Conv2D Work?

Each neuron in a `Conv2D` layer computes:

$$Y = \text{activation}(W * X + b)$$

Where:
- $X$ = Input data (image or feature map)
- $W$ = Set of learnable filters
- $*$ = Convolution operation
- $b$ = Bias term (optional)
- $\text{activation}$ = Activation function applied to the output

### 3.3 Pooling Layers in Keras

Pooling layers **reduce the spatial dimensions** of feature maps while retaining essential information. The most common types are **max pooling** and **average pooling**.

**Syntax:**
```python
from tensorflow.keras.layers import MaxPooling2D, AveragePooling2D

max_pool = MaxPooling2D(pool_size=(2, 2), strides=None, padding="valid")
avg_pool = AveragePooling2D(pool_size=(2, 2), strides=None, padding="valid")
```

### Main Arguments

| Argument | Description |
|---|---|
| `pool_size` | Size of the pooling window (e.g., `(2,2)`). |
| `strides` | Step size for sliding the pooling window. Default: Same as `pool_size`. |
| `padding` | `"valid"` (no padding) or `"same"` (zero-padding). |

### 3.4 How Does Pooling Work?

Pooling layers downsample feature maps to reduce computational complexity and control overfitting.

- **Max Pooling:** Selects the maximum value within each pooling window.
- **Average Pooling:** Computes the average of all values in each pooling window.

$$Y_{i,j} = \max(X_{m,n}) \quad \text{or} \quad Y_{i,j} = \frac{1}{N} \sum X_{m,n}$$

Where $X_{m,n}$ are the elements in the pooling window and $N$ is the number of elements in the window.

---

## 4. Training a Convolutional Neural Network

CNNs are trained using **forward** and **backward propagation**.

---

### 4.1 Forward Propagation

Forward propagation is the process in which input data passes through the network layer by layer until the final output layer.

**Steps:**

1. **Input Data Processing:**
   - The input data (image tensor) is passed through the network.
   - Each layer applies convolution, pooling, and dense operations.
   - Mathematical operations per layer:
     - *Convolutional Layer:* Convolution + ReLU activation
     - *Dense Layer:* $\sum WX + b$ + ReLU activation
     - *Output Layer:* $\sum W_{hl} + b$ + Softmax activation

2. **Final Output:**
   - The output layer generates a **probability distribution** over classes.
   - Output is compared to true labels using a **loss function** (Categorical Cross-Entropy).
   - The final cost is the **average loss** over all data points.

---

### 4.2 Backward Propagation

Backward propagation updates the model's weights to **minimize the loss**.

1. **Loss Calculation:** Computes the difference between predictions and ground truth.

2. **Gradient Calculation:** Gradients of the loss w.r.t. each weight are computed via the **chain rule**, layer by layer from the output:

   - **Output Layer (Fully Connected):**
   $$\delta^l = \frac{\partial L}{\partial z^l} = \hat{y} - y$$

   - **Hidden Layer (chain rule):**
   $$\delta^l = (W^T_{l+1} \cdot \delta^{l+1}) \odot a'(z)^l$$

   Where:
   - $\delta^l$ = gradient for layer $l$
   - $a'(z^l)$ = derivative of the activation function at layer $l$
   - $(W^T_{l+1} \cdot \delta^{l+1})$ = error propagated from the layer above

   - **Convolutional Layer Gradients:**
     - Before backpropagating into the conv layer, the gradient (a flattened vector) is **reshaped** back into a tensor matching the last feature map's dimensions.
     - **Gradient w.r.t. filter $F$:** Convolution between input $X$ and error gradient $\delta = \frac{\partial E}{\partial O}$
     - **Gradient w.r.t. input $X$:** Convolution between *flipped* filter $F$ and error gradient $\delta$

3. **Gradient Descent:** Weights are updated as:

$$w^l = w^l - \eta \, \Delta w$$
$$b^l = b^l - \eta \, \Delta b$$

The hyperparameter $\eta$ (learning rate) controls the step size of each update.

---

## 5. How Keras Handles Forward and Backward Propagation

In Keras, forward and backward propagation are **automatically managed** when you call the `fit()` method.

### 5.1 Model Compilation

The `compile()` method configures the model for training by specifying:
- **Optimizer** (e.g., `Adam`, `SGD`)
- **Loss function** (e.g., `sparse_categorical_crossentropy`)
- **Metrics** (e.g., `accuracy`)

In [ ]:
# Example: Model Compilation
# model.compile(optimizer='SGD', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

### 5.2 Model Training

The `fit()` method trains the model by iterating through the training data. For each batch, Keras automatically:

1. Performs the **forward pass** through all layers.
2. **Computes the loss** against ground truth labels.
3. **Back-propagates** the error to compute gradients.
4. **Updates weights** using the optimizer.

This repeats for multiple **epochs**, with the model progressively improving.

In [ ]:
# Example: Training the Model
# model.fit(train_ds, epochs=10, validation_data=val_ds)

---

## 6. End-to-End CNN Implementation using Keras

Below is a complete CNN implementation covering:
- Data loading & preprocessing
- Model definition
- Compilation
- Training
- Evaluation
- Prediction

### Step 1: Import Libraries

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

### Step 2: Load and Preprocess Data

In [ ]:
# Load the MNIST dataset
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Normalize pixel values to [0, 1]
x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32")  / 255.0

# Add channel dimension: (N, 28, 28) → (N, 28, 28, 1)
x_train = np.expand_dims(x_train, axis=-1)
x_test  = np.expand_dims(x_test,  axis=-1)

print(f"Training set shape : {x_train.shape}")
print(f"Test set shape     : {x_test.shape}")

### Step 3: Define the CNN Architecture

The network consists of the following layers:

| Layer | Output Shape | Parameters |
|---|---|---|
| `Conv2D` (32 filters, 3×3) | (26, 26, 32) | 320 |
| `MaxPooling2D` (2×2) | (13, 13, 32) | 0 |
| `Conv2D` (64 filters, 3×3) | (11, 11, 64) | 18,496 |
| `MaxPooling2D` (2×2) | (5, 5, 64) | 0 |
| `Flatten` | (1600,) | 0 |
| `Dense` (128 units, ReLU) | (128,) | 204,928 |
| `Dense` (10 units, Softmax) | (10,) | 1,290 |
| **Total** | | **225,034** |

In [ ]:
model = keras.Sequential([
    # ── Convolutional Block 1 ──────────────────────────────────────────────
    # 32 filters of size 3×3; valid padding → output: (26, 26, 32)
    layers.Conv2D(32, (3, 3), activation="relu", input_shape=(28, 28, 1)),
    # Max pooling 2×2, stride 2 → output: (13, 13, 32)
    layers.MaxPooling2D((2, 2)),

    # ── Convolutional Block 2 ──────────────────────────────────────────────
    # 64 filters of size 3×3; valid padding → output: (11, 11, 64)
    layers.Conv2D(64, (3, 3), activation="relu"),
    # Max pooling 2×2, stride 2 → output: (5, 5, 64)
    layers.MaxPooling2D((2, 2)),

    # ── Fully Connected Head ───────────────────────────────────────────────
    # Flatten: (5, 5, 64) → (1600,)
    layers.Flatten(),
    # Dense layer with 128 neurons and ReLU activation
    layers.Dense(128, activation="relu"),
    # Output layer: 10 neurons (one per digit class) with Softmax
    layers.Dense(10, activation="softmax")
])

model.summary()

### Step 4: Compile the Model

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

### Step 5: Train the Model

In [ ]:
history = model.fit(
    x_train, y_train,
    epochs=5,
    batch_size=32,
    validation_data=(x_test, y_test)
)

### Step 6: Evaluate the Model

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=2)
print(f"\nTest Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_acc:.4f}")

### Step 7: Make Predictions

In [ ]:
# Predict on the first 5 test samples
predictions = model.predict(x_test[:5])
predicted_labels = np.argmax(predictions, axis=1)

print("Predicted labels :", predicted_labels)
print("Actual labels    :", y_test[:5])

---

## 7. Layer-by-Layer Breakdown

### Layer 1 — Input
```
input_shape = (28, 28, 1)
```
- Grayscale images of size 28×28 with 1 channel.

---

### Layer 2 — First Convolutional Layer (`Conv2D`, 32 filters, 3×3)
- Applies **32 filters** of size 3×3 to detect low-level patterns (edges, textures).
- **ReLU** activation introduces non-linearity and avoids vanishing gradients.
- Output shape (valid padding):
$$\frac{28 - 3}{1} + 1 = 26 \quad \Rightarrow \quad (26, 26, 32)$$

---

### Layer 3 — First Pooling Layer (`MaxPooling2D`, 2×2)
- Reduces spatial dimensions by half using a 2×2 window with stride 2.
- Output shape:
$$\frac{26 - 2}{2} + 1 = 13 \quad \Rightarrow \quad (13, 13, 32)$$

---

### Layer 4 — Second Convolutional Layer (`Conv2D`, 64 filters, 3×3)
- Applies **64 filters** of size 3×3 to learn more complex features.
- Output shape (valid padding):
$$\frac{13 - 3}{1} + 1 = 11 \quad \Rightarrow \quad (11, 11, 64)$$

---

### Layer 5 — Second Pooling Layer (`MaxPooling2D`, 2×2)
- Further reduces spatial dimensions.
- Output shape:
$$\lfloor 11 / 2 \rfloor = 5 \quad \Rightarrow \quad (5, 5, 64)$$

---

### Layer 6 — Flatten
- Converts 3D feature maps into a 1D vector for the dense layers.
$$5 \times 5 \times 64 = 1600 \quad \Rightarrow \quad (1600,)$$

---

### Layer 7 — Fully Connected Dense Layer (128 units)
- Learns high-level combinations of features.
- **ReLU** activation.
- Output shape: $(128,)$

---

### Layer 8 — Output Layer (10 units)
- One neuron per digit class (0–9).
- **Softmax** activation converts raw scores to probabilities.
- Output shape: $(10,)$

---

### Parameter Count Summary

| Layer | Output Shape | Parameters |
|---|---|---|
| `Conv2D` (32, 3×3) | (26, 26, 32) | 320 |
| `MaxPooling2D` (2×2) | (13, 13, 32) | 0 |
| `Conv2D` (64, 3×3) | (11, 11, 64) | 18,496 |
| `MaxPooling2D` (2×2) | (5, 5, 64) | 0 |
| `Flatten` | (1600,) | 0 |
| `Dense` (128) | (128,) | 204,928 |
| `Dense` (10) | (10,) | 1,290 |
| **Total** | | **225,034** |